# Crypto price cleaning with yfinance

This notebook downloads daily market data for:
- VIX (`^VIX`)
- Ethereum (`ETH-USD`)
- Bitcoin (`BTC-USD`)

Then it cleans the relevant columns and exports a ready-to-use CSV file.

In [2]:
import pandas as pd
import yfinance as yf

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

In [3]:
# Configuration
TICKERS = {
    "VIX": "^VIX",
    "ETH": "ETH-USD",
    "BTC": "BTC-USD",
}

START_DATE = "2018-01-01"
END_DATE = None  # None means up to today

In [4]:
# Download price history from Yahoo Finance
df_raw = yf.download(
    tickers=list(TICKERS.values()),
    start=START_DATE,
    end=END_DATE,
    interval="1d",
    auto_adjust=False,
    group_by="ticker",
    progress=False,
)

if df_raw.empty:
    raise ValueError("No data was downloaded. Check internet connection or ticker symbols.")

print("Raw dataframe shape:", df_raw.shape)
df_raw.head()

Raw dataframe shape: (3020, 18)


Ticker           BTC-USD                                                                        ^VIX                                      \
Price               Open          High           Low         Close     Adj Close       Volume   Open   High   Low Close Adj Close Volume   
Date                                                                                                                                       
2018-01-01  14112.200195  14112.200195  13154.700195  13657.200195  13657.200195  10291200000    NaN    NaN   NaN   NaN       NaN    NaN   
2018-01-02  13625.000000  15444.599609  13163.599609  14982.099609  14982.099609  16846600192  10.95  11.07  9.52  9.77      9.77    0.0   
2018-01-03  14978.200195  15572.799805  14844.500000  15201.000000  15201.000000  16871900160   9.56   9.65  8.94  9.15      9.15    0.0   
2018-01-04  15270.700195  15739.700195  14522.200195  15599.200195  15599.200195  21783199744   9.01   9.31  8.92  9.22      9.22    0.0   
2018-01-05  15477.200195  17705.199219  15202.799805  17429.500000  17429.500000  23840899072   9.10   9.54  9.00  9.22      9.22    0.0   

Ticker         ETH-USD                                                               
Price             Open         High         Low       Close   Adj Close      Volume  
Date                                                                                 
2018-01-01  755.757019   782.530029  742.004028  772.640991  772.640991  2595760128  
2018-01-02  772.346008   914.830017  772.346008  884.443970  884.443970  5783349760  
2018-01-03  886.000000   974.471008  868.450989  962.719971  962.719971  5093159936  
2018-01-04  961.713013  1045.079956  946.085999  980.921997  980.921997  6502859776  
2018-01-05  975.750000  1075.390015  956.325012  997.719971  997.719971  6683149824

In [5]:
# Keep the adjusted close price and flatten column names
adj_close = df_raw.xs("Adj Close", axis=1, level=1)

symbol_to_name = {symbol: name for name, symbol in TICKERS.items()}
df_prices = adj_close.rename(columns=symbol_to_name)

# Ensure clean datetime index and sorted rows
df_prices.index = pd.to_datetime(df_prices.index, errors="coerce")
df_prices = df_prices[~df_prices.index.isna()]
df_prices = df_prices.sort_index()
df_prices = df_prices[~df_prices.index.duplicated(keep="last")]

# Ensure numeric dtype for modeling
df_prices = df_prices.apply(pd.to_numeric, errors="coerce")

print("Clean price dataframe shape:", df_prices.shape)
df_prices.head()

Clean price dataframe shape: (3020, 3)


Ticker,BTC,VIX,ETH
Date,,,
2018-01-01,13657.200195,NaN,772.640991
2018-01-02,14982.099609,9.77,884.443970
2018-01-03,15201.000000,9.15,962.719971
2018-01-04,15599.200195,9.22,980.921997
2018-01-05,17429.500000,9.22,997.719971


In [6]:
# Data quality checks
print("Date range:", df_prices.index.min().date(), "to", df_prices.index.max().date())
print("Columns:", df_prices.columns.tolist())
print("Missing values per column:")
print(df_prices.isna().sum())
print("Duplicate dates:", int(df_prices.index.duplicated().sum()))

Date range: 2018-01-01 to 2026-04-08
Columns: ['BTC', 'VIX', 'ETH']
Missing values per column:
Ticker
BTC      0
VIX    943
ETH      0
dtype: int64
Duplicate dates: 0


In [7]:
# Optional: keep only dates where all three assets are available
# Remove this line if you prefer to keep missing values.
df_final = df_prices.dropna(subset=["VIX", "ETH", "BTC"]).copy()

print("Final dataframe shape:", df_final.shape)
df_final.tail()

Final dataframe shape: (2077, 3)


Ticker,BTC,VIX,ETH
Date,,,
2026-04-01,68078.554688,24.540001,2138.737061
2026-04-02,66888.570312,23.870001,2056.852539
2026-04-06,68859.828125,24.170000,2107.761475
2026-04-07,71940.703125,25.780001,2241.806641
2026-04-08,70988.773438,21.040001,2182.987305


In [8]:
# Export cleaned prices to CSV
output_path = "crypto_prices_clean.csv"
df_final.to_csv(output_path, index=True)
print(f"Saved: {output_path}")

Saved: crypto_prices_clean.csv
